# 01 · A máquina que **vê** — detecção e contagem

Primeira demo de palco. Objetivo: sair do "IA é uma caixa de texto" e mostrar
que o modelo aponta **onde** está cada coisa, e **conta**.

Sequência sugerida no palco:
1. rodar numa foto comum → aparecem as caixas
2. mostrar a **contagem por classe** no painel
3. mexer na **confiança** e ver objetos aparecendo e sumindo → é aqui que a
   plateia entende o que significa "o modelo acha que"

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

# ── 2. a raiz de tudo, e a conferência de que ela é REAL ──────────────
#
# ARMADILHA que já custou uma sessão: a linha acima cria a variável
# `drive` (minúscula), que é o MÓDULO do Colab. Se algum caminho for
# escrito com `drive` em vez de `DRIVE`, o Python aceita numa boa e
# monta um caminho como
#     <module 'google.colab.drive' from '/usr/local/...'>/04-garrafas
# O código roda, cria pastas, exporta arquivos — tudo no disco
# temporário do Colab, que evapora quando a sessão encerra. Nada disso
# chega ao seu Drive, e não há erro nenhum na tela.
#
# A conferência abaixo transforma esse silêncio num aviso imediato.

DRIVE = "/content/drive/MyDrive/PALESTRA-IA"

if not DRIVE.startswith("/content/drive/"):
    raise SystemExit(
        "DRIVE aponta para fora do Google Drive: " + repr(DRIVE) + "\n"
        "Provavelmente algum caminho usou `drive` (o módulo) em vez de `DRIVE`.")
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("O Drive não montou. Rode esta célula de novo e autorize o acesso.")

os.makedirs(DRIVE, exist_ok=True)
print("raiz no Drive:", DRIVE)
print("existe de verdade:", os.path.isdir(DRIVE))

In [ ]:
# ── ajuste de PALCO: tudo grande, porque a sala enxerga de 6 a 10 m ──
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 110,
    "font.size": 22,
    "axes.titlesize": 30,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 22,
    "axes.grid": True,
    "grid.alpha": .25,
    "axes.facecolor": "#0d1117",
    "figure.facecolor": "#0d1117",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",
    "xtick.color": "#e6edf3",
    "ytick.color": "#e6edf3",
    "axes.edgecolor": "#30363d",
    "axes.titlecolor": "#3fe0a8",
})
VERDE, VERMELHO, CINZA = "#3fe0a8", "#ff5c5c", "#7d8590"
# DRIVE não é redefinido aqui de propósito: quem define é a célula de
# setup, e uma variável de caminho com duas origens é como se perde a
# noção de onde os arquivos foram parar.
print("palco configurado")

In [ ]:
# ── carrega o modelo (do Drive, sem depender da internet do local) ──
modelo = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")
print("classes que este modelo conhece:", len(modelo.names))
print(list(modelo.names.values()))

### Escolha a imagem

Se você já colocou fotos em `01-deteccao/entrada/`, a célula pega a primeira.
Se a pasta estiver vazia, ela usa uma imagem de exemplo da própria biblioteca —
a demo **nunca quebra por falta de arquivo**.

In [ ]:
import glob, os, cv2
entradas = sorted(glob.glob(f"{DRIVE}/01-deteccao/entrada/*"))
entradas = [e for e in entradas if e.lower().endswith((".jpg",".jpeg",".png",".webp"))]

if entradas:
    IMG = entradas[0]
else:
    IMG = "https://ultralytics.com/images/bus.jpg"   # reserva
print("usando:", IMG)

In [ ]:
# ── a inferencia: uma linha ──
r = modelo.predict(IMG, conf=0.25, verbose=False)[0]
im = r.plot(line_width=4, font_size=18)
print(f"{len(r.boxes)} objetos encontrados")

import matplotlib.pyplot as plt, cv2
plt.figure()
plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title('O que a máquina viu')
plt.tight_layout(); plt.show()

In [ ]:
# ── contagem por classe ──
from collections import Counter
nomes = [modelo.names[int(c)] for c in r.boxes.cls]
cont = Counter(nomes)
for classe, n in cont.most_common():
    print(f"  {n:>3}  {classe}")

In [ ]:
# ── o painel: a mesma contagem, do jeito que a sala enxerga ──
import matplotlib.pyplot as plt
itens = cont.most_common(8)
fig, ax = plt.subplots()
ax.barh([i[0] for i in itens][::-1], [i[1] for i in itens][::-1], color=VERDE, height=.62)
ax.set_title(f"{sum(cont.values())} objetos na cena")
ax.set_xlabel("quantidade")
for i, (nome, n) in enumerate(itens[::-1]):
    ax.text(n + .06, i, str(n), va="center", fontsize=26, color=VERDE, fontweight="bold")
ax.set_xlim(0, max(cont.values()) * 1.18)
plt.tight_layout(); plt.show()

### O momento didático: **confiança**

Toda caixa vem com um número — o quanto o modelo se compromete com aquilo.
Baixar a confiança faz aparecer mais coisa (inclusive erro); subir faz sobrar
só o que ele tem certeza. **Não existe valor certo: existe o custo de errar.**

> Fala de palco: *"num controle de estoque, deixar passar uma garrafa é
> chato. Num sensor de EPI, deixar passar um capacete é acidente. O mesmo
> botão, decisões opostas."*

In [ ]:
# ── o mesmo quadro em tres niveis de confianca ──
import matplotlib.pyplot as plt, cv2
NIVEIS = [0.10, 0.35, 0.70]
fig, axs = plt.subplots(1, 3, figsize=(22, 8))
for ax, c in zip(axs, NIVEIS):
    rr = modelo.predict(IMG, conf=c, verbose=False)[0]
    ax.imshow(cv2.cvtColor(rr.plot(line_width=4), cv2.COLOR_BGR2RGB))
    ax.set_title(f"conf {c:.2f}  ·  {len(rr.boxes)} objetos", fontsize=26)
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── guarda o resultado no Drive (para reusar no slide, se quiser) ──
import cv2, os
os.makedirs(f"{DRIVE}/01-deteccao/saida", exist_ok=True)
destino = f"{DRIVE}/01-deteccao/saida/deteccao.jpg"
cv2.imwrite(destino, im)
print("salvo em", destino)